# 🍎 Apple Dual-Model Training (Setup B)
Train **both** architectures on Apple diseases (4 classes, PlantVillage + Cornell AgriTech FGVC8):
1. **Model 1**: `EfficientNet-B3` (Higher-capacity compound scaled lesion detector, 300x300)
2. **Model 2**: `ResNet-50` (Deep residual texture extractor, 224x224)

**Classes (Exact Lab + Field Overlap):**
1. Apple — Apple Scab (*Venturia inaequalis*)
2. Apple — Black Rot / Frogeye Leaf Spot (*Botryosphaeria obtusa*)
3. Apple — Cedar Apple Rust (*Gymnosporangium juniperi-virginianae*)
4. Apple — Healthy Leaf ✅

**Runtime:** GPU (T4 recommended)

## Step 1: Check GPU

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('❌ Set Runtime -> Change runtime type -> T4 GPU')

## Step 2: Extract `splits_apple.zip` (From Google Drive or Local Upload)

In [ ]:
import os, glob

# Create extraction folder first
!mkdir -p /content/data/splits_apple

# 1. Search in Google Drive and local session storage
candidates = (
    glob.glob('/content/drive/MyDrive/**/splits_apple*.zip', recursive=True) +
    glob.glob('/content/drive/MyDrive/splits_apple*.zip') +
    glob.glob('/content/*splits_apple*.zip') +
    glob.glob('**/*splits_apple*.zip', recursive=True)
)
candidates = [c for c in candidates if os.path.isfile(c)]

if candidates:
    zip_path = candidates[0]
    print(f'✅ Found zip at: {zip_path} ({os.path.getsize(zip_path)/(1024*1024):.1f} MB)')
    print('Extracting files...')
    !unzip -q -o "$zip_path" -d /content/data/splits_apple
    print('\n✅ Extraction complete! Contents:')
    !ls /content/data/splits_apple
else:
    print('❌ splits_apple.zip not found yet!')
    print('Make sure Google Drive is mounted or upload splits_apple.zip.')

## Step 3: Shared Training Setup & Natural Background Weighting (2.0×)

In [ ]:
import time, copy, os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, models, transforms
from sklearn.metrics import classification_report, confusion_matrix
from tqdm.notebook import tqdm

# Locate data directory
DATA_DIR = '/content/data/splits_apple'
if not os.path.isdir(os.path.join(DATA_DIR, 'train')):
    for cand in ['/content/data/splits_apple/splits_apple', '/content/data', 'data/splits_apple']:
        if os.path.isdir(os.path.join(cand, 'train')):
            DATA_DIR = cand
            break

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CLASS_NAMES = sorted(os.listdir(os.path.join(DATA_DIR, 'train')))
NUM_CLASSES = len(CLASS_NAMES)
DISPLAY_NAMES = [c.replace('Apple___', 'Apple — ').replace('_', ' ') for c in CLASS_NAMES]
print(f'Using DATA_DIR: {DATA_DIR}')
print(f'Classes ({NUM_CLASSES}):', DISPLAY_NAMES)

for split in ['train', 'val', 'test']:
    tot = sum(len(os.listdir(os.path.join(DATA_DIR, split, c))) for c in CLASS_NAMES)
    print(f'  {split}: {tot} images')

def get_loaders(img_size, batch_size=32):
    train_tf = transforms.Compose([
        transforms.Resize((img_size + 24, img_size + 24)),
        transforms.RandomCrop(img_size),
        transforms.RandomHorizontalFlip(0.5),
        transforms.RandomVerticalFlip(0.3),
        transforms.RandomRotation(25),
        transforms.ColorJitter(0.3, 0.3, 0.2, 0.05),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        transforms.RandomErasing(p=0.15, scale=(0.02, 0.15)),
    ])
    eval_tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    
    train_ds = datasets.ImageFolder(os.path.join(DATA_DIR, 'train'), train_tf)
    val_ds   = datasets.ImageFolder(os.path.join(DATA_DIR, 'val'),   eval_tf)
    test_ds  = datasets.ImageFolder(os.path.join(DATA_DIR, 'test'),  eval_tf)
    
    counts = [0] * NUM_CLASSES
    for _, lbl in train_ds.samples:
        counts[lbl] += 1
    cw = [1.0 / max(c, 1) for c in counts]
    # 2.0x sampling weight for natural outdoor orchard images
    sw = [cw[lbl] * (2.0 if os.path.basename(p).startswith('field_') else 1.0) for p, lbl in train_ds.samples]
    sampler = WeightedRandomSampler(sw, len(sw), replacement=True)
    
    return (
        DataLoader(train_ds, batch_size=batch_size, sampler=sampler, num_workers=2, pin_memory=True),
        DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True),
        DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True),
    )

def train_model(model, train_loader, val_loader, epochs=10, lr=1e-3, desc='Training', save_path=None):
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr, weight_decay=1e-4)
    scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    best_acc = 0.0
    best_wts = copy.deepcopy(model.state_dict())
    
    print(f'\n=== {desc} ({epochs} epochs) ===')
    for ep in range(epochs):
        t0 = time.time()
        model.train()
        tr_loss, tr_cor, tr_tot = 0.0, 0, 0
        for x, y in tqdm(train_loader, desc=f'{desc} Epoch {ep+1}', leave=False):
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            tr_loss += loss.item() * x.size(0)
            _, p = torch.max(out, 1)
            tr_tot += y.size(0)
            tr_cor += (p == y).sum().item()
            
        model.eval()
        va_loss, va_cor, va_tot = 0.0, 0, 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                out = model(x)
                loss = criterion(out, y)
                va_loss += loss.item() * x.size(0)
                _, p = torch.max(out, 1)
                va_tot += y.size(0)
                va_cor += (p == y).sum().item()
        
        scheduler.step()
        val_acc = va_cor / va_tot
        star = ''
        if val_acc > best_acc:
            best_acc = val_acc
            best_wts = copy.deepcopy(model.state_dict())
            star = ' ⭐ BEST'
            if save_path:
                torch.save({'model_state_dict': best_wts}, save_path)
        print(f'Epoch {ep+1:>2}/{epochs} | Train {100*tr_cor/tr_tot:.1f}% | Val {100*val_acc:.1f}% | {time.time()-t0:.0f}s{star}')
        
    model.load_state_dict(best_wts)
    return model, best_acc

## Step 4: Train Model 1 — `EfficientNet-B3` 🍎

In [ ]:
tr_loader_eff, va_loader_eff, te_loader_eff = get_loaders(img_size=300, batch_size=32)

model_eff = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.DEFAULT)
for p in model_eff.features[:6].parameters():
    p.requires_grad = False
in_ftrs = model_eff.classifier[1].in_features
model_eff.classifier = nn.Sequential(
    nn.Dropout(0.4), nn.Linear(in_ftrs, 512), nn.SiLU(),
    nn.Dropout(0.2), nn.Linear(512, NUM_CLASSES)
)
model_eff = model_eff.to(DEVICE)

# 2-Phase Training
model_eff, _ = train_model(model_eff, tr_loader_eff, va_loader_eff, epochs=8, lr=1e-3, desc='EfficientNet-B3 Phase 1', save_path='apple_efficientnet_b3.pth')
for p in model_eff.parameters(): p.requires_grad = True
model_eff, eff_acc = train_model(model_eff, tr_loader_eff, va_loader_eff, epochs=6, lr=1e-4, desc='EfficientNet-B3 Phase 2', save_path='apple_efficientnet_b3.pth')

torch.save({'model_state_dict': model_eff.state_dict()}, 'apple_efficientnet_b3.pth')
print(f'\n✅ Model 1 (EfficientNet-B3) saved! Best Val Accuracy: {100*eff_acc:.2f}%')

## Step 5: Train Model 2 — `ResNet-50` 🍎

In [ ]:
tr_loader_res, va_loader_res, te_loader_res = get_loaders(img_size=224, batch_size=32)

model_res = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
for name, param in model_res.named_parameters():
    if 'layer3' not in name and 'layer4' not in name and 'fc' not in name:
        param.requires_grad = False
in_ftrs = model_res.fc.in_features
model_res.fc = nn.Sequential(
    nn.Dropout(0.4), nn.Linear(in_ftrs, 512), nn.ReLU(),
    nn.Dropout(0.2), nn.Linear(512, NUM_CLASSES)
)
model_res = model_res.to(DEVICE)

# 2-Phase Training
model_res, _ = train_model(model_res, tr_loader_res, va_loader_res, epochs=8, lr=1e-3, desc='ResNet-50 Phase 1', save_path='apple_resnet50.pth')
for p in model_res.parameters(): p.requires_grad = True
model_res, res_acc = train_model(model_res, tr_loader_res, va_loader_res, epochs=6, lr=1e-4, desc='ResNet-50 Phase 2', save_path='apple_resnet50.pth')

torch.save({'model_state_dict': model_res.state_dict()}, 'apple_resnet50.pth')
print(f'\n✅ Model 2 (ResNet-50) saved! Best Val Accuracy: {100*res_acc:.2f}%')

## Step 6: Test Set Evaluation & Consensus Ensemble

In [ ]:
model_eff.eval()
model_res.eval()

y_true, y_eff, y_res, y_ens = [], [], [], []

with torch.no_grad():
    for (x_eff, y), (x_res, _) in zip(te_loader_eff, te_loader_res):
        x_eff, x_res = x_eff.to(DEVICE), x_res.to(DEVICE)
        p_eff = torch.softmax(model_eff(x_eff), dim=1).cpu().numpy()
        p_res = torch.softmax(model_res(x_res), dim=1).cpu().numpy()
        p_ens = (p_eff + p_res) / 2.0
        
        y_true.extend(y.numpy())
        y_eff.extend(p_eff.argmax(axis=1))
        y_res.extend(p_res.argmax(axis=1))
        y_ens.extend(p_ens.argmax(axis=1))

y_true = np.array(y_true)
y_eff = np.array(y_eff)
y_res = np.array(y_res)
y_ens = np.array(y_ens)

acc_eff = (y_eff == y_true).mean() * 100
acc_res = (y_res == y_true).mean() * 100
acc_ens = (y_ens == y_true).mean() * 100
agree = (y_eff == y_res).mean() * 100

print('='*50)
print(f'📊 HELD-OUT TEST RESULTS (1,771 IMAGES):')
print(f'   Model 1 (EfficientNet-B3): {acc_eff:.2f}%')
print(f'   Model 2 (ResNet-50):       {acc_res:.2f}%')
print(f'   Dual Ensemble Consensus:   {acc_ens:.2f}%')
print(f'   Dual Model Agreement Rate: {agree:.2f}%')
print('='*50)
print('\nDetailed Ensemble Classification Report:')
print(classification_report(y_true, y_ens, target_names=DISPLAY_NAMES))

## Step 7: Download Both Apple Models ⬇️

In [ ]:
from google.colab import files
print('Downloading apple_efficientnet_b3.pth...')
files.download('apple_efficientnet_b3.pth')

print('Downloading apple_resnet50.pth...')
files.download('apple_resnet50.pth')

print('\nPlace both files in: d:\\gt\\models\\')